# QML-SleepNet — FINAL Metric-Champion Fixed Fusion Audit v1

## Why this notebook exists

The project now has two strong but materially different frozen Task-A systems:

1. **Stage15A physiology challenger** — 379-D physiology → nested boosting → temperature → HMM.
2. **Guide-primary QML system** — Bridge + QT Angle-Rx + corrected Stage06, equal-logit fusion of their strict post-HMM scores.

The current project priority is **highest legitimate metric first**, while leakage control and scientific honesty remain non-negotiable.

This notebook performs **one final, zero-training, development-only fusion audit** to test whether the two systems contain complementary errors.

## Guide/pipeline boundary

This exact CatBoost↔QML fusion is **not an original guide-prescribed Stage06 block**.

- The original guide-primary architecture remains QML8 + CNN-BiLSTM + causal16 → fusion → classifier.
- Later project/guide-extension work contains a leakage-safe Stage15D fusion concept.
- Therefore this notebook is an **optional metric-first auxiliary fusion audit**, not a claim that the original guide explicitly prescribed CatBoost+QML fusion.

## Hard scientific boundary

**No official x labels are loaded anywhere in this notebook.**

The official x probabilities used here were frozen previously, before their respective scorers were run. Their file hashes are verified, but no x label file/path is opened.

The two parent models have already been scored historically on x elsewhere in the project. Therefore any later fused x result is necessarily **post-hoc historical-benchmark evidence**, not pristine untouched external validation.

## Predeclared candidate family

No continuous weight optimization. No threshold search. No calibration refit. No HMM refit. No retraining.

Only three fixed logit-space fusions:

- 25% physiology + 75% QML
- 50% physiology + 50% QML
- 75% physiology + 25% QML

Both parent models are also reported.

### Promotion gate

A fusion can be frozen for later x scoring only if it beats the **development-best parent by accuracy** and passes all of:

- accuracy gain ≥ **+0.10 percentage points**
- balanced-accuracy loss ≤ **0.10 pp**
- F1 loss ≤ **0.10 pp**
- AUPRC loss ≤ **0.10 pp**
- sensitivity loss ≤ **0.50 pp**

The gate is intentionally practical rather than a significance claim. Strict-group cluster bootstrap uncertainty is reported separately.

If no fixed fusion passes, **stop metric experimentation** and retain the already-frozen 91.0192% Stage15A physiology system as the project metric champion.

In [1]:
# Cell 1 — imports, Drive and immutable paths
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, sys, uuid, warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss
)

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
CONFIG_DIR = ROOT / "config"

# Stage15A physiology challenger — exact v1.2 coverage-safe run.
CAT_ROOT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "DEV35_STAGE15A_PHYSIO_BOOSTING_CHALLENGER_v1_2_COVERAGESAFE"
)
CAT_OOF = CAT_ROOT / "DEV35_STRICT_V2_OOF_PREDICTIONS.npz"
CAT_DEV_METRICS = CAT_ROOT / "DEV35_STRICT_V2_GLOBAL_METRICS.json"
CAT_X = CAT_ROOT / "DEV35_STAGE15A_PHYSIO_BOOSTING_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
CAT_FREEZE = CAT_ROOT / "DEV35_STAGE15A_PHYSIO_BOOSTING_PRE_SCORE_FREEZE_MANIFEST.json"

# Guide-primary QML system — strict development OOF and frozen x prediction.
QML_STRICT_ROOT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "EXISTING_ARTIFACT_STRICT_OUTER_FUSION_CORRECTION_v1"
)
QML_OOF = QML_STRICT_ROOT / "STRICT_OUTER_OOF_SCORES.npz"
QML_DECISION = QML_STRICT_ROOT / "STRICT_OUTER_FINAL_DECISION.json"

QML_X_ROOT = ROOT / "outputs/GUIDE_EXACT_METRICMAX/FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
QML_X = QML_X_ROOT / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"

STRICT_V2 = ROOT / "outputs/strict_protocol/strict_learn_5fold_balanced_v2.json"

OUT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1"
)
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_CAT_X_SHA = "e548af5c1d8ad39f8ea192680ee7ae7af303b993e65f410ea2ee18f015e3e244"
EXPECTED_QML_X_SHA = "f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef"
EXPECTED_DEV35_V2_SHA = "be078c7cec97fbcf2d9cd6e46f6c2d82cf278323375a9b2bd933792dfeb86464"

EPS = 1e-8
THRESHOLD = 0.5

FIXED_PHYS_WEIGHTS = (0.25, 0.50, 0.75)

GATE = {
    "accuracy_gain_min": 0.0010,       # +0.10 pp
    "balanced_accuracy_loss_max": 0.0010,
    "f1_loss_max": 0.0010,
    "auprc_loss_max": 0.0010,
    "sensitivity_loss_max": 0.0050,
}

for p in [CAT_OOF, CAT_DEV_METRICS, CAT_X, CAT_FREEZE,
          QML_OOF, QML_DECISION, QML_X, STRICT_V2]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("ZERO TRAINING")
print("ZERO OFFICIAL-X LABEL ACCESS")
print("Fixed physiology fusion weights:", FIXED_PHYS_WEIGHTS)


Mounted at /content/drive
ZERO TRAINING
ZERO OFFICIAL-X LABEL ACCESS
Fixed physiology fusion weights: (0.25, 0.5, 0.75)


In [2]:
# Cell 2 — helpers
def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + f".tmp.{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)

def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_name(path.stem + f".tmp.{uuid.uuid4().hex}.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)

def logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1-EPS)
    return np.log(p) - np.log1p(-p)

def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def fixed_logit_fusion(p_phys, p_qml, w_phys):
    w = float(w_phys)
    return sigmoid(w*logit(p_phys) + (1.0-w)*logit(p_qml))

def metric_row(y, score):
    y = np.asarray(y, dtype=np.int8)
    score = np.asarray(score, dtype=np.float64)
    pred = (score >= THRESHOLD).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn/max(tn+fp,1)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "auroc": float(roc_auc_score(y, score)),
        "auprc": float(average_precision_score(y, score)),
        "brier": float(brier_score_loss(y, score)),
        "nll": float(log_loss(y, np.column_stack([1-score,score]), labels=[0,1])),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def parse_record(uid):
    return str(uid).rsplit(":", 1)[0]

def candidate_rank(row):
    return (
        -row["accuracy"],
        -row["balanced_accuracy"],
        -row["f1"],
        -row["auprc"],
        -row["auroc"],
        row["candidate"],
    )


In [3]:
# Cell 3 — verify immutable lineages and the no-label development contract
# DEV35 strict lock.
split = json.loads(STRICT_V2.read_text())
stored = split.get("lock_sha256")
core = dict(split)
core.pop("lock_sha256", None)
recomputed = hashlib.sha256(
    json.dumps(core, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
if stored != EXPECTED_DEV35_V2_SHA or recomputed != EXPECTED_DEV35_V2_SHA:
    raise RuntimeError("DEV35_STRICT_V2 lock mismatch")

# Stage15A pre-score freeze.
cat_freeze = json.loads(CAT_FREEZE.read_text())
if cat_freeze.get("official_x_labels_used_for_selection") is not False:
    raise RuntimeError("Stage15A provenance says x labels influenced selection")
if cat_freeze.get("official_x_labels_read_in_this_notebook") is not False:
    raise RuntimeError("Stage15A development notebook read x labels")
if cat_freeze.get("stage15a_target_index_y_loaded") is not False:
    raise RuntimeError("Stage15A target-index y was loaded")
if cat_freeze.get("prediction_sha256") != EXPECTED_CAT_X_SHA:
    raise RuntimeError("Stage15A pre-score manifest x SHA mismatch")
if sha256_file(CAT_X) != EXPECTED_CAT_X_SHA:
    raise RuntimeError("Stage15A frozen x file SHA mismatch")

# QML strict decision / frozen x.
qdec = json.loads(QML_DECISION.read_text())
if qdec.get("official_x_labels_used") is not False:
    raise RuntimeError("QML strict decision provenance failure")
if sha256_file(QML_X) != EXPECTED_QML_X_SHA:
    raise RuntimeError("QML frozen x SHA mismatch")

print("DEV35_STRICT_V2 lock:", EXPECTED_DEV35_V2_SHA)
print("Stage15A frozen x SHA:", EXPECTED_CAT_X_SHA)
print("QML frozen x SHA:", EXPECTED_QML_X_SHA)
print("OFFICIAL X LABELS LOADED: NO")


DEV35_STRICT_V2 lock: be078c7cec97fbcf2d9cd6e46f6c2d82cf278323375a9b2bd933792dfeb86464
Stage15A frozen x SHA: e548af5c1d8ad39f8ea192680ee7ae7af303b993e65f410ea2ee18f015e3e244
QML frozen x SHA: f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef
OFFICIAL X LABELS LOADED: NO


In [4]:
# Cell 4 — load and UID-align the two strict development OOF systems
cat = np.load(CAT_OOF, allow_pickle=False)
CAT_UID = np.asarray(cat["uids"]).astype(str)
CAT_Y = np.asarray(cat["y"], dtype=np.int8)
P_PHYS = np.asarray(cat["hmm_apnea_posterior"], dtype=np.float64)

q = np.load(QML_OOF, allow_pickle=False)
Q_UID = np.asarray(q["uids"]).astype(str)
Q_Y = np.asarray(q["y"], dtype=np.int8)
Q_FOLD = np.asarray(q["fold"], dtype=np.int8)

required_q = {"bridge", "qt_angle_rx", "current_stage06"}
if not required_q.issubset(q.files):
    raise RuntimeError(f"QML OOF missing keys {required_q-set(q.files)}")

if len(np.unique(CAT_UID)) != len(CAT_UID):
    raise RuntimeError("Stage15A OOF duplicate UID")
if len(np.unique(Q_UID)) != len(Q_UID):
    raise RuntimeError("QML OOF duplicate UID")
if set(CAT_UID) != set(Q_UID):
    raise RuntimeError("Stage15A/QML DEV35 UID universes differ")

q_index = {u:i for i,u in enumerate(Q_UID)}
qi = np.asarray([q_index[u] for u in CAT_UID], dtype=np.int64)

if not np.array_equal(CAT_Y, Q_Y[qi]):
    raise RuntimeError("Stage15A/QML development labels disagree")

Q_FOLD_ALIGNED = Q_FOLD[qi]
P_QML_PARTS = {
    "bridge": np.asarray(q["bridge"], dtype=np.float64)[qi],
    "qt_angle_rx": np.asarray(q["qt_angle_rx"], dtype=np.float64)[qi],
    "current_stage06": np.asarray(q["current_stage06"], dtype=np.float64)[qi],
}
for name, arr in P_QML_PARTS.items():
    if len(arr) != len(CAT_Y) or not np.isfinite(arr).all():
        raise RuntimeError(f"{name}: invalid QML strict OOF score")

P_QML = sigmoid(np.mean(
    np.column_stack([logit(P_QML_PARTS[k]) for k in
                     ["bridge","qt_angle_rx","current_stage06"]]),
    axis=1
))

if not np.isfinite(P_PHYS).all() or not np.isfinite(P_QML).all():
    raise RuntimeError("Non-finite parent OOF score")
if np.any((P_PHYS < 0) | (P_PHYS > 1) | (P_QML < 0) | (P_QML > 1)):
    raise RuntimeError("Parent OOF score outside [0,1]")

UID = CAT_UID
Y = CAT_Y
REC = np.asarray([parse_record(u) for u in UID])

print("Aligned DEV35 rows:", len(UID))
print("Apnea prevalence:", float(Y.mean()))
print("QML strict OOF fold counts:", dict(zip(*np.unique(Q_FOLD_ALIGNED, return_counts=True))))


Aligned DEV35 rows: 17023
Apnea prevalence: 0.382482523644481
QML strict OOF fold counts: {np.int8(0): np.int64(2966), np.int8(1): np.int64(3893), np.int8(2): np.int64(3398), np.int8(3): np.int64(3433), np.int8(4): np.int64(3333)}


In [5]:
# Cell 5 — verify/compare the QML OOF fold map against the immutable DEV35_STRICT_V2 record map
v2_fold_by_record = {}
for f in core["folds"]:
    fold = int(f["fold"])
    for r in f["val_records"]:
        if r in v2_fold_by_record:
            raise RuntimeError(f"Duplicate DEV35 v2 validation record {r}")
        v2_fold_by_record[r] = fold

if set(v2_fold_by_record) != set(np.unique(REC)):
    raise RuntimeError("DEV35 v2 validation record coverage != OOF record universe")

V2_FOLD = np.asarray([v2_fold_by_record[r] for r in REC], dtype=np.int8)
fold_match = (V2_FOLD == Q_FOLD_ALIGNED)

fold_audit = {
    "n_rows": int(len(UID)),
    "qml_fold_matches_dev35_v2_rows": int(fold_match.sum()),
    "qml_fold_differs_dev35_v2_rows": int((~fold_match).sum()),
    "match_fraction": float(fold_match.mean()),
    "fixed_fusion_allowed_if_different": True,
    "learned_meta_weight_allowed_if_different": False,
}
atomic_json(OUT / "FOLD_SYSTEM_ALIGNMENT_AUDIT.json", fold_audit)
print(json.dumps(fold_audit, indent=2))

if not fold_match.all():
    print(
        "NOTE: OOF fold systems differ for some rows. This notebook remains valid only "
        "because it uses fixed, predeclared fusion weights and does NOT fit a second-level learner."
    )


{
  "n_rows": 17023,
  "qml_fold_matches_dev35_v2_rows": 932,
  "qml_fold_differs_dev35_v2_rows": 16091,
  "match_fraction": 0.05474945661751748,
  "fixed_fusion_allowed_if_different": true,
  "learned_meta_weight_allowed_if_different": false
}
NOTE: OOF fold systems differ for some rows. This notebook remains valid only because it uses fixed, predeclared fusion weights and does NOT fit a second-level learner.


In [6]:
# Cell 6 — parent metrics, disagreement and complementarity
M_PHYS = metric_row(Y, P_PHYS)
M_QML = metric_row(Y, P_QML)

parent_rows = [
    {"candidate": "PARENT_STAGE15A_PHYSIOLOGY", **M_PHYS},
    {"candidate": "PARENT_GUIDE_QML_CORE3", **M_QML},
]
parent_df = pd.DataFrame(parent_rows).sort_values(
    ["accuracy","balanced_accuracy","f1","auprc","auroc"],
    ascending=False
).reset_index(drop=True)
parent_df.to_csv(OUT / "PARENT_DEV35_METRICS.csv", index=False)
display(parent_df)

PRED_PHYS = (P_PHYS >= 0.5).astype(np.int8)
PRED_QML = (P_QML >= 0.5).astype(np.int8)

phys_correct = PRED_PHYS == Y
qml_correct = PRED_QML == Y

comp = {
    "n": int(len(Y)),
    "prediction_disagreement_n": int(np.sum(PRED_PHYS != PRED_QML)),
    "prediction_disagreement_fraction": float(np.mean(PRED_PHYS != PRED_QML)),
    "both_correct_n": int(np.sum(phys_correct & qml_correct)),
    "phys_only_correct_n": int(np.sum(phys_correct & ~qml_correct)),
    "qml_only_correct_n": int(np.sum(~phys_correct & qml_correct)),
    "both_wrong_n": int(np.sum(~phys_correct & ~qml_correct)),
    "logit_pearson_r": float(np.corrcoef(logit(P_PHYS), logit(P_QML))[0,1]),
}
atomic_json(OUT / "PARENT_COMPLEMENTARITY.json", comp)
print(json.dumps(comp, indent=2))

DEV_PARENT_NAME = parent_df.iloc[0]["candidate"]
DEV_PARENT_SCORE = P_PHYS if DEV_PARENT_NAME == "PARENT_STAGE15A_PHYSIOLOGY" else P_QML
DEV_PARENT_METRICS = M_PHYS if DEV_PARENT_NAME == "PARENT_STAGE15A_PHYSIOLOGY" else M_QML

print("\nDEVELOPMENT-BEST PARENT:", DEV_PARENT_NAME)


,candidate,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,mcc,auroc,auprc,brier,nll,tn,fp,fn,tp
0,PARENT_GUIDE_QML_CORE3,17023,0.888151,0.874597,0.881943,0.816925,0.932268,0.848190,0.761250,0.941538,0.921381,0.099816,0.6122,9800,712,1192,5319
1,PARENT_STAGE15A_PHYSIOLOGY,17023,0.867591,0.845351,0.885668,0.750730,0.939973,0.812635,0.717139,0.941742,0.920429,0.117076,0.6624,9881,631,1623,4888


{
  "n": 17023,
  "prediction_disagreement_n": 1728,
  "prediction_disagreement_fraction": 0.10150972214063327,
  "both_correct_n": 14080,
  "phys_only_correct_n": 689,
  "qml_only_correct_n": 1039,
  "both_wrong_n": 1215,
  "logit_pearson_r": 0.8830891539193179
}

DEVELOPMENT-BEST PARENT: PARENT_GUIDE_QML_CORE3


In [7]:
# Cell 7 — evaluate the ONLY three predeclared fixed fusion candidates
candidate_scores = {}
rows = []

for w in FIXED_PHYS_WEIGHTS:
    name = f"FIXED_LOGIT_PHYS_{int(round(100*w)):02d}_QML_{int(round(100*(1-w))):02d}"
    score = fixed_logit_fusion(P_PHYS, P_QML, w)
    candidate_scores[name] = score
    m = metric_row(Y, score)
    rows.append({
        "candidate": name,
        "w_phys": float(w),
        "w_qml": float(1-w),
        **m,
        "accuracy_delta_vs_dev_best_parent_pp":
            100*(m["accuracy"] - DEV_PARENT_METRICS["accuracy"]),
        "balanced_accuracy_delta_vs_dev_best_parent_pp":
            100*(m["balanced_accuracy"] - DEV_PARENT_METRICS["balanced_accuracy"]),
        "f1_delta_vs_dev_best_parent_pp":
            100*(m["f1"] - DEV_PARENT_METRICS["f1"]),
        "auprc_delta_vs_dev_best_parent_pp":
            100*(m["auprc"] - DEV_PARENT_METRICS["auprc"]),
        "sensitivity_delta_vs_dev_best_parent_pp":
            100*(m["sensitivity"] - DEV_PARENT_METRICS["sensitivity"]),
    })

fusion_df = pd.DataFrame(rows).sort_values(
    ["accuracy","balanced_accuracy","f1","auprc","auroc"],
    ascending=False
).reset_index(drop=True)
fusion_df.to_csv(OUT / "FIXED_FUSION_DEV35_CANDIDATES.csv", index=False)
display(fusion_df)


,candidate,w_phys,w_qml,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,...,nll,tn,fp,fn,tp,accuracy_delta_vs_dev_best_parent_pp,balanced_accuracy_delta_vs_dev_best_parent_pp,f1_delta_vs_dev_best_parent_pp,auprc_delta_vs_dev_best_parent_pp,sensitivity_delta_vs_dev_best_parent_pp
0,FIXED_LOGIT_PHYS_25_QML_75,0.25,0.75,17023,0.895083,0.881115,0.895397,0.821686,0.940544,0.856960,...,0.541260,9887,625,1161,5350,0.693180,0.651871,0.876971,0.958086,0.476117
1,FIXED_LOGIT_PHYS_50_QML_50,0.50,0.50,17023,0.894848,0.879025,0.903573,0.811703,0.946347,0.855178,...,0.515698,9948,564,1226,5285,0.669682,0.442861,0.698791,1.306028,-0.522193
2,FIXED_LOGIT_PHYS_75_QML_25,0.75,0.25,17023,0.881572,0.861611,0.899982,0.776686,0.946537,0.833800,...,0.557595,9950,562,1454,5057,-0.657933,-1.298509,-1.438959,1.003252,-4.023959


In [8]:
# Cell 8 — predeclared promotion gate; NO adaptive rescue if it fails
def gate_result(m):
    return {
        "accuracy_gain_pass": bool(
            m["accuracy"] - DEV_PARENT_METRICS["accuracy"] >= GATE["accuracy_gain_min"]
        ),
        "balanced_accuracy_pass": bool(
            m["balanced_accuracy"] >=
            DEV_PARENT_METRICS["balanced_accuracy"] - GATE["balanced_accuracy_loss_max"]
        ),
        "f1_pass": bool(
            m["f1"] >= DEV_PARENT_METRICS["f1"] - GATE["f1_loss_max"]
        ),
        "auprc_pass": bool(
            m["auprc"] >= DEV_PARENT_METRICS["auprc"] - GATE["auprc_loss_max"]
        ),
        "sensitivity_pass": bool(
            m["sensitivity"] >=
            DEV_PARENT_METRICS["sensitivity"] - GATE["sensitivity_loss_max"]
        ),
    }

gated = []
for row in rows:
    g = gate_result(row)
    passed = all(g.values())
    gated.append({**row, **g, "promotion_gate_pass": passed})

gate_df = pd.DataFrame(gated).sort_values(
    ["promotion_gate_pass","accuracy","balanced_accuracy","f1","auprc","auroc"],
    ascending=[False,False,False,False,False,False]
).reset_index(drop=True)
gate_df.to_csv(OUT / "FIXED_FUSION_PROMOTION_GATE.csv", index=False)
display(gate_df)

passing = [r for r in gated if r["promotion_gate_pass"]]
if passing:
    WIN = sorted(passing, key=candidate_rank)[0]
    PROMOTE = True
else:
    WIN = None
    PROMOTE = False

print("PROMOTE_FUSION:", PROMOTE)
if PROMOTE:
    print("DEVELOPMENT WINNER:", WIN["candidate"])
else:
    print("No fixed fusion passed. No x fusion will be created.")


,candidate,w_phys,w_qml,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,...,balanced_accuracy_delta_vs_dev_best_parent_pp,f1_delta_vs_dev_best_parent_pp,auprc_delta_vs_dev_best_parent_pp,sensitivity_delta_vs_dev_best_parent_pp,accuracy_gain_pass,balanced_accuracy_pass,f1_pass,auprc_pass,sensitivity_pass,promotion_gate_pass
0,FIXED_LOGIT_PHYS_25_QML_75,0.25,0.75,17023,0.895083,0.881115,0.895397,0.821686,0.940544,0.856960,...,0.651871,0.876971,0.958086,0.476117,True,True,True,True,True,True
1,FIXED_LOGIT_PHYS_50_QML_50,0.50,0.50,17023,0.894848,0.879025,0.903573,0.811703,0.946347,0.855178,...,0.442861,0.698791,1.306028,-0.522193,True,True,True,True,False,False
2,FIXED_LOGIT_PHYS_75_QML_25,0.75,0.25,17023,0.881572,0.861611,0.899982,0.776686,0.946537,0.833800,...,-1.298509,-1.438959,1.003252,-4.023959,False,False,False,True,False,False


PROMOTE_FUSION: True
DEVELOPMENT WINNER: FIXED_LOGIT_PHYS_25_QML_75


In [9]:
# Cell 9 — strict-group cluster bootstrap of each fusion's accuracy delta vs development-best parent
if str(CONFIG_DIR) not in sys.path:
    sys.path.insert(0, str(CONFIG_DIR))
for mod in ["qml_sleepnet_strict_protocol"]:
    if mod in sys.modules:
        del sys.modules[mod]
import qml_sleepnet_strict_protocol as strict

GROUP = np.asarray([strict.strict_group(r) for r in REC], dtype="U64")
groups = np.unique(GROUP)
idx_by_group = {g: np.where(GROUP == g)[0] for g in groups}

rng = np.random.default_rng(20260914)
B = 5000
boot_rows = []

base_pred = (DEV_PARENT_SCORE >= 0.5).astype(np.int8)
base_acc = accuracy_score(Y, base_pred)

for name, score in candidate_scores.items():
    pred = (score >= 0.5).astype(np.int8)
    observed = float(accuracy_score(Y, pred) - base_acc)
    draws = np.empty(B, dtype=np.float64)

    for b in range(B):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        ii = np.concatenate([idx_by_group[g] for g in sampled])
        draws[b] = (
            accuracy_score(Y[ii], pred[ii]) -
            accuracy_score(Y[ii], base_pred[ii])
        )

    boot_rows.append({
        "candidate": name,
        "observed_accuracy_delta_pp": 100*observed,
        "bootstrap_mean_delta_pp": 100*float(np.mean(draws)),
        "ci95_low_pp": 100*float(np.quantile(draws, 0.025)),
        "ci95_high_pp": 100*float(np.quantile(draws, 0.975)),
        "bootstrap_probability_delta_gt_0": float(np.mean(draws > 0)),
        "bootstrap_replicates": B,
        "bootstrap_unit": "strict subject/source group",
    })

boot_df = pd.DataFrame(boot_rows).sort_values(
    "observed_accuracy_delta_pp", ascending=False
).reset_index(drop=True)
boot_df.to_csv(OUT / "FIXED_FUSION_STRICT_GROUP_BOOTSTRAP.csv", index=False)
display(boot_df)


,candidate,observed_accuracy_delta_pp,bootstrap_mean_delta_pp,ci95_low_pp,ci95_high_pp,bootstrap_probability_delta_gt_0,bootstrap_replicates,bootstrap_unit
0,FIXED_LOGIT_PHYS_25_QML_75,0.693180,0.691146,0.224746,1.128619,0.9978,5000,strict subject/source group
1,FIXED_LOGIT_PHYS_50_QML_50,0.669682,0.684907,-1.230237,1.997629,0.7972,5000,strict subject/source group
2,FIXED_LOGIT_PHYS_75_QML_25,-0.657933,-0.628245,-5.318378,2.305619,0.4070,5000,strict subject/source group


In [10]:
# Cell 10 — record-level audit for the winning fixed candidate or best fixed candidate
if PROMOTE:
    AUDIT_NAME = WIN["candidate"]
else:
    AUDIT_NAME = fusion_df.iloc[0]["candidate"]

AUDIT_SCORE = candidate_scores[AUDIT_NAME]

record_rows = []
for r in sorted(np.unique(REC)):
    m = REC == r
    pm = metric_row(Y[m], AUDIT_SCORE[m])
    bm = metric_row(Y[m], DEV_PARENT_SCORE[m])
    record_rows.append({
        "record_name": r,
        "n": int(m.sum()),
        "candidate": AUDIT_NAME,
        "candidate_accuracy": pm["accuracy"],
        "dev_parent_accuracy": bm["accuracy"],
        "accuracy_delta_pp": 100*(pm["accuracy"]-bm["accuracy"]),
        "candidate_sensitivity": pm["sensitivity"],
        "candidate_specificity": pm["specificity"],
    })

record_df = pd.DataFrame(record_rows).sort_values("accuracy_delta_pp")
record_df.to_csv(OUT / "FUSION_RECORD_LEVEL_AUDIT.csv", index=False)
display(record_df)


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metric

,record_name,n,candidate,candidate_accuracy,dev_parent_accuracy,accuracy_delta_pp,candidate_sensitivity,candidate_specificity
8,a09,495,FIXED_LOGIT_PHYS_25_QML_75,0.858586,0.884848,-2.626263,0.845144,0.903509
17,a18,489,FIXED_LOGIT_PHYS_25_QML_75,0.848671,0.862986,-1.431493,0.860731,0.745098
6,a07,510,FIXED_LOGIT_PHYS_25_QML_75,0.792157,0.803922,-1.176471,0.968847,0.492063
7,a08,500,FIXED_LOGIT_PHYS_25_QML_75,0.774000,0.782000,-0.800000,0.534392,0.919614
23,b04,428,FIXED_LOGIT_PHYS_25_QML_75,0.976636,0.981308,-0.467290,0.000000,1.000000
10,a11,466,FIXED_LOGIT_PHYS_25_QML_75,0.761803,0.766094,-0.429185,0.518018,0.983607
27,c03,453,FIXED_LOGIT_PHYS_25_QML_75,0.988962,0.991170,-0.220751,0.000000,0.988962
0,a01,489,FIXED_LOGIT_PHYS_25_QML_75,0.983640,0.983640,0.000000,1.000000,0.578947
20,b01,486,FIXED_LOGIT_PHYS_25_QML_75,0.960905,0.960905,0.000000,0.000000,1.000000
31,c07,428,FIXED_LOGIT_PHYS_25_QML_75,0.985981,0.985981,0.000000,0.000000,0.995283


In [11]:
# Cell 11 — if and ONLY if the development gate passes, fuse the already-frozen x probabilities and freeze before scoring
FUSED_X_PATH = OUT / "FINAL_FIXED_FUSION_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
FUSION_FREEZE = OUT / "FINAL_FIXED_FUSION_PRE_SCORE_FREEZE_MANIFEST.json"

if PROMOTE:
    # Load x probabilities only. No x labels exist anywhere in this notebook.
    cx = np.load(CAT_X, allow_pickle=False)
    qx = np.load(QML_X, allow_pickle=False)

    C_UID = np.asarray(cx["test_uids"]).astype(str)
    C_SCORE = np.asarray(cx["hmm_apnea_posterior"], dtype=np.float64)

    Q_UID = np.asarray(qx["test_uids"]).astype(str)
    Q_SCORE = np.asarray(qx["ensemble_hmm_logit_mean"], dtype=np.float64)

    if len(np.unique(C_UID)) != len(C_UID) or len(np.unique(Q_UID)) != len(Q_UID):
        raise RuntimeError("Duplicate x UID")
    if set(C_UID) != set(Q_UID):
        raise RuntimeError("Parent x UID universes differ")

    qmap = {u:i for i,u in enumerate(Q_UID)}
    qix = np.asarray([qmap[u] for u in C_UID], dtype=np.int64)
    Q_SCORE = Q_SCORE[qix]

    if not np.isfinite(C_SCORE).all() or not np.isfinite(Q_SCORE).all():
        raise RuntimeError("Non-finite parent x score")

    w = float(WIN["w_phys"])
    FUSED_X = fixed_logit_fusion(C_SCORE, Q_SCORE, w)
    FUSED_PRED = (FUSED_X >= THRESHOLD).astype(np.int8)

    atomic_npz(
        FUSED_X_PATH,
        test_uids=C_UID.astype("U128"),
        physiology_hmm_posterior=C_SCORE.astype(np.float64),
        qml_core3_hmm_logit_mean=Q_SCORE.astype(np.float64),
        physiology_weight=np.asarray(w, dtype=np.float64),
        qml_weight=np.asarray(1-w, dtype=np.float64),
        fused_probability=FUSED_X.astype(np.float64),
        prediction=FUSED_PRED.astype(np.int8),
        hard_threshold=np.asarray(THRESHOLD, dtype=np.float64),
    )
    fused_sha = sha256_file(FUSED_X_PATH)

    manifest = {
        "status": "FIXED_FUSION_FROZEN_BEFORE_ANY_FUSION_X_SCORING",
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "development_best_parent": DEV_PARENT_NAME,
        "development_best_parent_metrics": DEV_PARENT_METRICS,
        "fixed_candidate_family": [
            {"w_phys": float(w0), "w_qml": float(1-w0)}
            for w0 in FIXED_PHYS_WEIGHTS
        ],
        "promotion_gate": GATE,
        "winning_fusion": WIN,
        "stage15a_component_x_sha256": EXPECTED_CAT_X_SHA,
        "qml_component_x_sha256": EXPECTED_QML_X_SHA,
        "fused_prediction_file": str(FUSED_X_PATH),
        "fused_prediction_sha256": fused_sha,
        "n_x_rows": int(len(FUSED_X)),
        "hard_threshold": THRESHOLD,
        "training_performed": False,
        "calibration_refit": False,
        "hmm_refit": False,
        "threshold_search": False,
        "official_x_labels_loaded": False,
        "scientific_status": (
            "post-hoc development extension conceived after historical component x scores "
            "were known; fusion method itself selected only from strict development OOF"
        ),
        "guide_status": (
            "optional metric-first auxiliary fusion; not an original guide-prescribed Stage06 block"
        ),
    }
    atomic_json(FUSION_FREEZE, manifest)

    print("="*110)
    print("FUSION FROZEN BEFORE SCORING")
    print("Winner:", WIN["candidate"])
    print("Fused x SHA256:", fused_sha)
    print("Rows:", len(FUSED_X))
    print("OFFICIAL X LABELS LOADED: NO")
    print("="*110)
else:
    # Fail closed: never create a fused x bundle if development gate fails.
    if FUSED_X_PATH.exists():
        raise RuntimeError(
            "A fused x file already exists from another run, but this run did not pass the gate. "
            "Do not score it; inspect lineage manually."
        )
    print("DEVELOPMENT GATE FAILED — CURRENT 91.0192% STAGE15A METRIC CHAMPION REMAINS LOCKED.")


FUSION FROZEN BEFORE SCORING
Winner: FIXED_LOGIT_PHYS_25_QML_75
Fused x SHA256: 063a017e61188393bcdcdacb72958ffa3e7e0efa9432d33aa0845983462dfa1f
Rows: 17248
OFFICIAL X LABELS LOADED: NO


In [12]:
# Cell 12 — final development decision artifact
decision = {
    "audit": "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "priority": "highest legitimate metric first",
    "official_x_labels_loaded": False,
    "training_performed": False,
    "candidate_family": "three fixed post-HMM logit fusions only",
    "fixed_phys_weights": list(FIXED_PHYS_WEIGHTS),
    "development_best_parent": DEV_PARENT_NAME,
    "development_best_parent_metrics": DEV_PARENT_METRICS,
    "parent_phys_metrics": M_PHYS,
    "parent_qml_metrics": M_QML,
    "parent_complementarity": comp,
    "gate": GATE,
    "promote_fusion": PROMOTE,
    "winner": WIN,
    "fold_system_alignment": fold_audit,
    "next_action": (
        "Run separate frozen fusion x scorer exactly once"
        if PROMOTE else
        "Stop metric experimentation; refresh XAI for Stage15A champion then proceed to Stage10 robustness"
    ),
}
atomic_json(OUT / "FINAL_FIXED_FUSION_DEVELOPMENT_DECISION.json", decision)

print("\nFINAL DEVELOPMENT DECISION")
print(json.dumps(decision, indent=2))



FINAL DEVELOPMENT DECISION
{
  "audit": "FINAL_METRIC_CHAMPION_FIXED_FUSION_AUDIT_v1",
  "created_utc": "2026-09-13T20:59:57.602931+00:00",
  "priority": "highest legitimate metric first",
  "official_x_labels_loaded": false,
  "training_performed": false,
  "candidate_family": "three fixed post-HMM logit fusions only",
  "fixed_phys_weights": [
    0.25,
    0.5,
    0.75
  ],
  "development_best_parent": "PARENT_GUIDE_QML_CORE3",
  "development_best_parent_metrics": {
    "n": 17023,
    "accuracy": 0.8881513246783763,
    "balanced_accuracy": 0.8745965439122225,
    "precision": 0.8819432929862377,
    "sensitivity": 0.8169252035017662,
    "specificity": 0.9322678843226788,
    "f1": 0.8481900813267421,
    "mcc": 0.7612499226966759,
    "auroc": 0.9415381989079714,
    "auprc": 0.9213814968613496,
    "brier": 0.0998162640148414,
    "nll": 0.612199726261056,
    "tn": 9800,
    "fp": 712,
    "fn": 1192,
    "tp": 5319
  },
  "parent_phys_metrics": {
    "n": 17023,
    "accurac

## STOP

If `PROMOTE_FUSION: False`, do **not** run another weight search. The metric campaign is closed and the 91.0192% Stage15A physiology system remains champion.

If `PROMOTE_FUSION: True`, this notebook freezes an x prediction NPZ without labels. Send the executed notebook/output back for audit **before running the separate scorer**.